# Biohub Cell Tracking — этап 1: detector

Нормальный воспроизводимый training detector центров клеток. Linker, граф и submission пока не запускаются. Split: целые видео одного эмбриона → независимый второй эмбрион.


## 1. Конфигурация и окружение

По умолчанию длительное обучение и validation выключены. Сначала проверьте split и один реальный smoke-step.


In [ ]:
from __future__ import annotations
import json, math, os, random, sys, time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import tracksdata as td
import zarr
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd().resolve()
VENDOR = ROOT / "vendor" / "kaggle-cell-tracking-competition"
if not (VENDOR / "src").exists(): raise FileNotFoundError(VENDOR)
sys.path.insert(0, str(VENDOR / "src"))
from tracking_cellmot.models import TemporalUNet3D

@dataclass
class Config:
    # Порядок эмбрионов можно поменять для обратного независимого fold.
    train_prefix: str = "44b6"
    val_prefix: str = "6bba"
    data_root: str = str(ROOT)
    run_name: str = "detector_44b6_to_6bba_v2"
    output_root: str = str(ROOT / "outputs")
    seed: int = 42
    window_size: int = 2
    downsample: tuple[int, int, int] = (1, 4, 4)
    batch_size: int = 1              # variable-size GT; не увеличивать без padded collate.
    grad_accum_steps: int = 4
    num_workers: int = 0
    max_epochs: int = 3
    learning_rate: float = 1e-4
    gradient_clip_norm: float = 1.0
    use_amp: bool = True
    amp_initial_scale: float = 1024.0
    unet_layers: tuple[int, int, int] = (32, 64, 128)
    unet_out_channels: int = 32
    gradient_checkpointing: bool = False
    negative_weight: float = 2.0     # Суммарная масса unknown/background относительно positives; >1 защищает от all-positive collapse.
    initial_probability: float = 0.01
    nms_um: float = 5.0
    detection_threshold_probability: float = 0.50
    match_distance_um: float = 7.0
    resume_path: str | None = None
    detector_checkpoint_path: str | None = None  # Для validation: None означает RUN/last.pt.
    validate_every_n_epochs: int = 1  # Полный holdout после каждой N-й эпохи, если run_validation=True.
    live_plot_every_steps: int = 50  # Как часто обновлять график в output ячейки обучения.
    run_training: bool = False       # Явно включить только после проверки ячеек ниже.
    run_validation: bool = False

cfg = Config()
if cfg.train_prefix == cfg.val_prefix: raise ValueError("train и validation должны быть разными группами эмбрионов")
if cfg.window_size != 2: raise ValueError("Текущий detector поддерживает только window_size=2")
if cfg.batch_size != 1: raise ValueError("Текущий collate поддерживает только batch_size=1")
if cfg.grad_accum_steps < 1: raise ValueError("grad_accum_steps должен быть положительным")
if cfg.amp_initial_scale <= 0: raise ValueError("amp_initial_scale должен быть положительным")
if cfg.negative_weight <= 1: raise ValueError("negative_weight должен быть >1, иначе постоянный all-positive output выгоднее threshold=0.5")
if not 0 < cfg.initial_probability < cfg.detection_threshold_probability < 1: raise ValueError("Нужны 0 < initial_probability < detection_threshold_probability < 1")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = cfg.use_amp and device.type == "cuda"
RUN = Path(cfg.output_root) / cfg.run_name
RUN.mkdir(parents=True, exist_ok=True)
if cfg.run_training and cfg.resume_path is None and (RUN / "last.pt").exists():
    raise FileExistsError(f"{RUN / 'last.pt'} уже существует. Для нового запуска смените run_name; для продолжения задайте resume_path.")
(RUN / "config.json").write_text(json.dumps(asdict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
if device.type == "cuda": torch.cuda.manual_seed_all(cfg.seed)
print({"device": str(device), "amp": amp_enabled, "run_dir": str(RUN)})


## 2. Аудит и split

Обучение на `44b6`, независимая validation на `6bba`; для обратного fold поменяйте префиксы.


In [ ]:
TRAIN = Path(cfg.data_root) / "train"

def ids(folder: Path) -> list[str]:
    return sorted(p.stem for p in folder.glob("*.zarr") if (folder / f"{p.stem}.geff").exists())

def load_graph(path: Path):
    value = td.graph.IndexedRXGraph.from_geff(path)
    return value[0] if isinstance(value, tuple) else value

def image_scale(group) -> tuple[float, float, float]:
    return tuple(dict(group.attrs)["multiscales"][0]["datasets"][0]["coordinateTransformations"][0]["scale"][-3:])

def graph_nodes(path: Path) -> pl.DataFrame:
    return load_graph(path).node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])

all_ids = ids(TRAIN)
train_ids = [x for x in all_ids if x.startswith(cfg.train_prefix)]
val_ids = [x for x in all_ids if x.startswith(cfg.val_prefix)]
if not train_ids or not val_ids: raise ValueError("Не найдены обе группы из конфигурации")
if set(train_ids) & set(val_ids): raise ValueError("Split пересекается")
split = {"train_prefix": cfg.train_prefix, "validation_prefix": cfg.val_prefix, "train": train_ids, "validation": val_ids}
(RUN / "split.json").write_text(json.dumps(split, ensure_ascii=False, indent=2), encoding="utf-8")

audit = []
for dataset_id in all_ids:
    group = zarr.open_group(TRAIN / f"{dataset_id}.zarr", mode="r")
    array = group["0"]; nodes = graph_nodes(TRAIN / f"{dataset_id}.geff")
    if array.ndim != 4: raise ValueError(f"{dataset_id}: ожидается TZYX")
    if nodes["node_id"].n_unique() != nodes.height: raise ValueError(f"{dataset_id}: duplicate node IDs")
    if nodes["t"].min() < 0 or nodes["t"].max() >= array.shape[0]: raise ValueError(f"{dataset_id}: time out of bounds")
    audit.append({"dataset": dataset_id, "group": dataset_id.split("_")[0], "shape": tuple(array.shape), "chunks": tuple(array.chunks),
                  "scale_zyx_um": image_scale(group), "sparse_nodes": nodes.height})
audit = pd.DataFrame(audit)
audit.to_csv(RUN / "data_audit.csv", index=False)
display(audit.groupby("group").agg(videos=("dataset","count"), sparse_nodes=("sparse_nodes","sum")))


## 3. Ленивый Dataset

Читается только пара кадров. Координаты переводятся в downsample-grid ровно один раз.


In [ ]:
def stored_quantiles(group) -> tuple[float, float]:
    quantiles = dict(group.attrs).get("image_statistics", {}).get("quantiles", {})
    if "0.001" in quantiles and "0.999" in quantiles:
        return float(quantiles["0.001"]), float(quantiles["0.999"])
    sample = np.asarray(group["0"][0, ::2, ::8, ::8], dtype=np.float32)
    return tuple(float(v) for v in np.quantile(sample, [0.001, 0.999]))

def normalize(raw: np.ndarray, q_low: float, q_high: float) -> np.ndarray:
    return np.clip((raw.astype(np.float32) - q_low) / max(q_high - q_low, 1e-6), 0, 1)

class DetectionWindows(Dataset):
    """Лениво читает только два кадра; набор изображений целиком не материализуется."""
    def __init__(self, folder: Path, dataset_ids: list[str], downsample: tuple[int,int,int]):
        self.downsample = np.asarray(downsample, dtype=np.float32)
        self.entries = []
        for dataset_id in dataset_ids:
            zpath = folder / f"{dataset_id}.zarr"; group = zarr.open_group(zpath, mode="r")
            n_frames = group["0"].shape[0]; q_low, q_high = stored_quantiles(group)
            nodes = graph_nodes(folder / f"{dataset_id}.geff")
            for t in range(n_frames - 1):
                a = nodes.filter(pl.col("t") == t).select(["z","y","x"])
                b = nodes.filter(pl.col("t") == t + 1).select(["z","y","x"])
                # Окно без размеченных центров не обучает detector и пропускается.
                if a.height or b.height:
                    self.entries.append((dataset_id, zpath, t, a, b, q_low, q_high))
    def __len__(self): return len(self.entries)
    def __getitem__(self, index: int) -> dict:
        dataset_id, zpath, t, a, b, q_low, q_high = self.entries[index]
        dz, dy, dx = self.downsample.astype(int)
        raw = np.asarray(zarr.open_group(zpath, mode="r")["0"][t:t+2, ::dz, ::dy, ::dx], dtype=np.float32)
        coords = lambda frame: torch.from_numpy(frame.to_numpy().astype(np.float32) / self.downsample)
        return {"dataset": dataset_id, "t": t, "images": torch.from_numpy(normalize(raw, q_low, q_high)),
                "coords": (coords(a), coords(b))}

train_dataset = DetectionWindows(TRAIN, train_ids, cfg.downsample)
print(f"Train windows with sparse labels: {len(train_dataset)}; sample: {tuple(train_dataset[0]['images'].shape)}")


## 4. Detector и sparse loss

Неаннотированные воксели получают слабый отрицательный вес, а не объявляются достоверным фоном.


In [ ]:
class TemporalDetector(nn.Module):
    """Официальный TemporalUNet3D и голова центров; вход (B,T,Z,Y,X)."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.unet = TemporalUNet3D(1, cfg.unet_out_channels, cfg.unet_layers,
                                   gradient_checkpointing=cfg.gradient_checkpointing)
        self.head = nn.Conv3d(cfg.unet_out_channels, 1, kernel_size=1)
        nn.init.constant_(self.head.bias, math.log(cfg.initial_probability / (1 - cfg.initial_probability)))
    def forward(self, images: torch.Tensor) -> list[torch.Tensor]:
        features = self.unet(images.unsqueeze(2))              # (B,T,C,Z,Y,X)
        return [self.head(features[:, t]) for t in range(features.shape[1])]  # T × (B,1,Z,Y,X)

def weighted_detection_loss(logits: torch.Tensor, coords: torch.Tensor, negative_weight: float) -> torch.Tensor:
    """Sparse-GT heatmap loss: GT-centres positive, unknown voxels слабо штрафуются."""
    target = torch.zeros_like(logits[:, 0])
    if len(coords):
        z, y, x = [coords[:, i].long().clamp(0, target.shape[i+1]-1) for i in range(3)]
        target[0, z, y, x] = 1.0
    loss_map = F.binary_cross_entropy_with_logits(logits[:, 0].float(), target.float(), reduction="none")
    positive = target > 0
    negative_loss = loss_map[~positive].mean()
    if not len(coords):
        return negative_loss
    positive_loss = loss_map[positive].mean()
    # Нормируем классы отдельно: масштаб loss не зависит от 3D-объёма.
    return (positive_loss + negative_weight * negative_loss) / (1 + negative_weight)

model = TemporalDetector(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled, init_scale=cfg.amp_initial_scale)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 5. Smoke-step

Один forward/backward/optimizer step для проверки loss, градиентов и CUDA. Это не цикл обучения.


In [ ]:
# Короткая проверка вычислимости. Это один шаг, не обучение по эпохе.
def one_training_step(batch: dict) -> tuple[torch.Tensor, dict]:
    images = batch["images"].unsqueeze(0).to(device, non_blocking=True)
    coords = [x.to(device, dtype=torch.float32) for x in batch["coords"]]
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type, enabled=amp_enabled):
        logits = model(images)
        loss = sum(weighted_detection_loss(x, c, cfg.negative_weight) for x, c in zip(logits, coords))
    if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite detection loss: {loss.item()}")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip_norm))
    if not math.isfinite(grad_norm): raise FloatingPointError(f"Non-finite gradient norm in smoke-step: {grad_norm}")
    scaler.step(optimizer); scaler.update()
    return loss.detach(), {"loss": float(loss.detach().cpu()), "grad_norm": grad_norm,
                           "nodes_t": len(coords[0]), "nodes_t1": len(coords[1])}

smoke_loss, smoke_log = one_training_step(train_dataset[0])
print(smoke_log)
# Модель изменилась только в smoke-step; перед полноценным train создаётся заново в следующей ячейке.


## 6. Обучение и resume

Явно установите `cfg.run_training=True`. Сохраняются `last.pt`, config, split и числовая история.


In [ ]:
def save_checkpoint(path: Path, epoch: int, step: int, best_recall: float) -> None:
    state = {"detector_checkpoint_version": 2, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
             "epoch": epoch, "step": step, "best_sparse_recall": best_recall, "config": asdict(cfg),
             "train_ids": train_ids, "val_ids": val_ids, "rng_torch": torch.get_rng_state(),
             "rng_numpy": np.random.get_state(), "rng_python": random.getstate(),
             "rng_cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None}
    torch.save(state, path)

def load_checkpoint(path: Path) -> tuple[int, int, float]:
    state = torch.load(path, map_location=device, weights_only=False)
    if state.get("detector_checkpoint_version") != 2: raise ValueError("Checkpoint создан старой версией detector; начните новый run")
    current = asdict(cfg); saved = state.get("config", {})
    critical = ("train_prefix", "val_prefix", "downsample", "unet_layers", "unet_out_channels",
                "negative_weight", "initial_probability")
    mismatched = [key for key in critical if saved.get(key) != current.get(key)]
    if mismatched: raise ValueError(f"Checkpoint несовместим с текущей конфигурацией: {mismatched}")
    if state.get("train_ids") != train_ids or state.get("val_ids") != val_ids:
        raise ValueError("Checkpoint создан для другого train/validation manifest")
    model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"]); scaler.load_state_dict(state["scaler"])
    torch.set_rng_state(state["rng_torch"].cpu())
    np.random.set_state(state["rng_numpy"]); random.setstate(state["rng_python"])
    if device.type == "cuda" and state.get("rng_cuda") is not None:
        torch.cuda.set_rng_state_all([value.cpu() for value in state["rng_cuda"]])
    return int(state["epoch"]) + 1, int(state["step"]), float(state.get("best_sparse_recall", -1))

def _live_training_svg(losses: list[float], validation_points: list[tuple[int, float]]) -> str:
    width, height, left, right = 900, 430, 55, 25
    if len(losses) > 2000:
        keep = np.linspace(0, len(losses) - 1, 2000, dtype=int); losses = [losses[i] for i in keep]
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="white"/>', '<style>text{font:14px sans-serif;fill:#172033}.title{font-size:17px;font-weight:600}.grid{stroke:#e5e7eb}.loss{fill:none;stroke:#2563eb;stroke-width:2}.recall{fill:none;stroke:#059669;stroke-width:2}</style>']
    def panel(values, title, top, color, maximum=None):
        chart_h = 145; bottom = top + chart_h; parts.append(f'<text class="title" x="{left}" y="{top - 10}">{title}</text>')
        for fraction in (0, .5, 1): parts.append(f'<line class="grid" x1="{left}" y1="{top + fraction * chart_h:.1f}" x2="{width-right}" y2="{top + fraction * chart_h:.1f}"/>')
        if len(values) < 2: parts.append(f'<text x="{left}" y="{top + 75}">Ожидание данных…</text>'); return
        ys = np.asarray([v for _, v in values], dtype=float); lo, hi = float(ys.min()), float(ys.max()) if maximum is None else maximum
        if hi <= lo: hi = lo + 1.0
        xs = np.linspace(left, width-right, len(ys)); py = bottom - (ys - lo) / (hi - lo) * chart_h
        points = ' '.join(f'{x:.1f},{y:.1f}' for x, y in zip(xs, py)); parts.append(f'<polyline class="{color}" points="{points}"/>')
        parts.append(f'<text x="{left}" y="{bottom + 18}">min {lo:.4f} · max {hi:.4f} · последнее {ys[-1]:.4f}</text>')
    panel(list(enumerate(losses)), 'Train loss — обновляется во время эпохи', 45, 'loss')
    panel(validation_points, 'Validation sparse-GT recall — после каждой validation', 260, 'recall', maximum=1.0)
    return ''.join(parts) + '</svg>'

def _finish_accumulation(accumulated: int) -> float:
    scaler.unscale_(optimizer)
    for parameter in model.parameters():
        if parameter.grad is not None: parameter.grad.div_(accumulated)
    grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip_norm))
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    return grad_norm

def _persist_training_artifacts(rows: list[dict], epoch_rows: list[dict]) -> None:
    pd.DataFrame(rows).to_csv(RUN / "train_history.csv", index=False)
    pd.DataFrame(epoch_rows).to_csv(RUN / "epoch_history.csv", index=False)
    runtime = {"device": str(device), "amp": amp_enabled, "torch_version": torch.__version__,
               "epochs_completed": len(epoch_rows), "total_seconds": float(sum(x["seconds"] for x in epoch_rows))}
    if device.type == "cuda": runtime.update({"gpu_name": torch.cuda.get_device_name(device), "gpu_total_memory_gib": torch.cuda.get_device_properties(device).total_memory / 2**30})
    (RUN / "training_runtime.json").write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding="utf-8")

def train_detector() -> pd.DataFrame:
    global model, optimizer, scaler
    # Новый init исключает влияние smoke-step.
    model = TemporalDetector(cfg).to(device); optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled, init_scale=cfg.amp_initial_scale)
    start_epoch = step = 0; best_recall = -1.0
    if cfg.resume_path:
        start_epoch, step, best_recall = load_checkpoint(Path(cfg.resume_path))
    loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,
                        collate_fn=lambda items: items[0], pin_memory=(device.type=="cuda"))
    from IPython.display import SVG, display
    history_source = Path(cfg.resume_path).resolve().parent if cfg.resume_path else RUN
    rows = pd.read_csv(history_source / 'train_history.csv').to_dict(orient='records') if cfg.resume_path and (history_source / 'train_history.csv').exists() else []
    epoch_rows = pd.read_csv(history_source / 'epoch_history.csv').to_dict(orient='records') if cfg.resume_path and (history_source / 'epoch_history.csv').exists() else []
    model.train(); optimizer.zero_grad(set_to_none=True)
    previous_validation = pd.read_csv(history_source / 'validation_history.csv') if cfg.resume_path and (history_source / 'validation_history.csv').exists() else pd.DataFrame()
    validation_rows = previous_validation.to_dict(orient='records') if len(previous_validation) else []
    live_validation = [(int(row['epoch']), float(row['sparse_recall'])) for row in validation_rows]
    live_handle = display(SVG(_live_training_svg([], live_validation)), display_id=True)
    for epoch in tqdm(range(start_epoch, cfg.max_epochs), desc='Detector: эпохи', position=0):
        epoch_started = time.perf_counter(); epoch_losses = []; accumulated = 0
        if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
        progress = tqdm(loader, total=len(loader), desc=f'Эпоха {epoch + 1}/{cfg.max_epochs}', leave=False, position=1)
        for batch in progress:
            images = batch["images"].unsqueeze(0).to(device, non_blocking=True)
            coords = [x.to(device, dtype=torch.float32, non_blocking=True) for x in batch["coords"]]
            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(images)
                loss = sum(weighted_detection_loss(x, c, cfg.negative_weight) for x, c in zip(logits, coords))
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss at step {step}")
            scaler.scale(loss).backward(); accumulated += 1; grad_norm = np.nan
            if accumulated == cfg.grad_accum_steps:
                grad_norm = _finish_accumulation(accumulated); accumulated = 0
            loss_value = float(loss.detach().cpu()); epoch_losses.append(loss_value)
            rows.append({"epoch": epoch, "step": step, "loss": loss_value,
                         "nodes_t": len(coords[0]), "nodes_t1": len(coords[1]),
                         "learning_rate": optimizer.param_groups[0]["lr"], "grad_norm": grad_norm})
            step += 1
            if step % cfg.live_plot_every_steps == 0 or step == 1:
                live = {"loss": f'{loss_value:.4f}', "mean": f'{np.mean(epoch_losses):.4f}'}
                if device.type == 'cuda': live['VRAM GiB'] = f'{torch.cuda.memory_allocated(device) / 2**30:.2f}'
                progress.set_postfix(live)
                live_svg = _live_training_svg([r['loss'] for r in rows], live_validation)
                (RUN / 'live_training.svg').write_text(live_svg, encoding='utf-8')
                if live_handle is not None: live_handle.update(SVG(live_svg))
        if accumulated:
            rows[-1]["grad_norm"] = _finish_accumulation(accumulated)
        epoch_row = {"epoch": epoch, "steps": len(epoch_losses), "mean_loss": float(np.mean(epoch_losses)),
                     "median_loss": float(np.median(epoch_losses)), "seconds": time.perf_counter() - epoch_started,
                     "learning_rate": optimizer.param_groups[0]["lr"],
                     "peak_gpu_memory_gib": (torch.cuda.max_memory_allocated(device) / 2**30 if device.type == "cuda" else 0.0)}
        epoch_rows.append(epoch_row); print(epoch_row)
        save_checkpoint(RUN / "last.pt", epoch, step, best_recall)
        _persist_training_artifacts(rows, epoch_rows)  # История не теряется, даже если validation завершится ошибкой.
        if cfg.run_validation and (epoch + 1) % cfg.validate_every_n_epochs == 0:
            print(f'Эпоха {epoch + 1}: независимая validation на {len(val_ids)} видео…')
            validate_detector(load_checkpoint=False); model.train()
            validation_summary = json.loads((RUN / 'detector_validation_summary.json').read_text(encoding='utf-8'))
            validation_row = {"epoch": epoch, **validation_summary}
            validation_rows = [row for row in validation_rows if int(row['epoch']) != epoch] + [validation_row]
            pd.DataFrame(validation_rows).sort_values('epoch').to_csv(RUN / 'validation_history.csv', index=False)
            epoch_row.update({"validation_sparse_recall": validation_summary['sparse_recall'], "validation_localisation_error_um": validation_summary['mean_localisation_error_um'], "validation_seconds": validation_summary['validation_seconds']})
            current_recall = float(validation_summary['sparse_recall'])
            if np.isfinite(current_recall) and current_recall > best_recall:
                best_recall = current_recall; save_checkpoint(RUN / 'best.pt', epoch, step, best_recall)
            save_checkpoint(RUN / 'last.pt', epoch, step, best_recall)
            live_validation = [point for point in live_validation if point[0] != epoch] + [(epoch, current_recall)]
            live_svg = _live_training_svg([r['loss'] for r in rows], live_validation)
            (RUN / 'live_training.svg').write_text(live_svg, encoding='utf-8')
            if live_handle is not None: live_handle.update(SVG(live_svg))
        _persist_training_artifacts(rows, epoch_rows)
    return pd.DataFrame(rows)

# Запуск оставлен после определения validation ниже: это позволяет строить validation-кривую между эпохами.


## 7. Независимая detector validation

Metrics: sparse-GT recall, средняя физическая ошибка локализации, число peaks. Precision на sparse GT не выдаётся за полный precision.


In [ ]:
def _physical_nms(coords: np.ndarray, scores: np.ndarray, effective_scale_zyx: np.ndarray) -> np.ndarray:
    """Greedy score-ordered NMS with an exact Euclidean radius in micrometres."""
    if not len(coords): return np.empty(0, dtype=np.int64)
    physical = coords.astype(np.float64) * effective_scale_zyx.astype(np.float64)
    tree = cKDTree(physical); order = np.argsort(-scores, kind='stable')
    suppressed = np.zeros(len(coords), dtype=bool); keep = []
    radius2 = cfg.nms_um ** 2
    for index in order:
        if suppressed[index]: continue
        keep.append(index)
        neighbours = np.asarray(tree.query_ball_point(physical[index], r=cfg.nms_um), dtype=np.int64)
        delta = physical[neighbours] - physical[index]
        suppressed[neighbours[(delta * delta).sum(axis=1) < radius2]] = True
    return np.asarray(keep, dtype=np.int64)

@torch.inference_mode()
def peaks(logits: torch.Tensor, scale_zyx: tuple[float,float,float]) -> torch.Tensor:
    """One point per local-max plateau, followed by physical-distance NMS."""
    probability = torch.sigmoid(logits[0, 0].float())
    pooled = F.max_pool3d(probability[None, None], 3, stride=1, padding=1)[0, 0]
    mask = (probability == pooled) & (probability >= cfg.detection_threshold_probability)
    if not mask.any(): return torch.empty((0, 3), device=logits.device, dtype=torch.float32)
    probability_np = probability.cpu().numpy(); mask_np = mask.cpu().numpy()
    labels, count = ndimage.label(mask_np, structure=np.ones((3, 3, 3), dtype=bool))
    positions = ndimage.maximum_position(probability_np, labels, np.arange(1, count + 1))
    coords = np.asarray(positions, dtype=np.float32).reshape(-1, 3)
    scores = probability_np[tuple(coords.astype(np.int64).T)]
    effective_scale = np.asarray(scale_zyx, dtype=np.float32) * np.asarray(cfg.downsample, dtype=np.float32)
    keep = _physical_nms(coords, scores, effective_scale)
    return torch.as_tensor(coords[keep], device=logits.device, dtype=torch.float32)

def match_sparse(pred: torch.Tensor, gt: torch.Tensor, scale_zyx: tuple[float,float,float]) -> list[float]:
    """Maximum-cardinality one-to-one matching, with distance as the tie-breaker."""
    if not len(pred) or not len(gt): return []
    effective_scale = np.asarray(scale_zyx, dtype=np.float64) * np.asarray(cfg.downsample, dtype=np.float64)
    pred_np = pred.detach().cpu().numpy().astype(np.float64) * effective_scale
    gt_np = gt.detach().cpu().numpy().astype(np.float64) * effective_scale
    distance = np.linalg.norm(pred_np[:, None, :] - gt_np[None, :, :], axis=2)
    penalty = (min(distance.shape) + 1) * (cfg.match_distance_um + 1)
    rows, columns = linear_sum_assignment(np.where(distance <= cfg.match_distance_um, distance, penalty))
    matched = distance[rows, columns]; return matched[matched <= cfg.match_distance_um].astype(float).tolist()

def load_detector_for_validation() -> Path:
    global model
    checkpoint = Path(cfg.detector_checkpoint_path) if cfg.detector_checkpoint_path else RUN / 'last.pt'
    if not checkpoint.exists(): raise FileNotFoundError(f'Нет checkpoint для validation: {checkpoint}. Сначала включите cfg.run_training=True.')
    state = torch.load(checkpoint, map_location=device, weights_only=False)
    if state.get('detector_checkpoint_version') != 2: raise ValueError('Checkpoint создан старой версией detector; обучите v2 заново')
    saved = state.get('config', {}); current = asdict(cfg)
    critical = ('train_prefix', 'val_prefix', 'downsample', 'unet_layers', 'unet_out_channels')
    mismatched = [key for key in critical if saved.get(key) != current.get(key)]
    if mismatched or state.get('train_ids') != train_ids or state.get('val_ids') != val_ids:
        raise ValueError(f'Checkpoint несовместим с validation split/config: {mismatched}')
    model = TemporalDetector(cfg).to(device); model.load_state_dict(state['model']); model.eval()
    return checkpoint

@torch.inference_mode()
def validate_detector(load_checkpoint: bool = True) -> pd.DataFrame:
    checkpoint = load_detector_for_validation() if load_checkpoint else RUN / 'last.pt'; model.eval(); rows = []; validation_started = time.perf_counter()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats(device)
    for dataset_id in tqdm(val_ids, desc='Validation: видео', position=0):
        zpath = TRAIN / f"{dataset_id}.zarr"; group = zarr.open_group(zpath, mode="r"); array = group["0"]
        qlow, qhigh = stored_quantiles(group); scale_zyx = image_scale(group)
        gt = graph_nodes(TRAIN / f"{dataset_id}.geff")
        matched_all, n_gt, n_pred = [], 0, 0; dataset_started = time.perf_counter()
        # Для t=0 берём первый output первого окна; далее второй output каждого окна.
        for t in tqdm(range(array.shape[0]-1), desc=f'Validation {dataset_id}', leave=False, position=1):
            raw = np.asarray(array[t:t+2, ::cfg.downsample[0], ::cfg.downsample[1], ::cfg.downsample[2]], dtype=np.float32)
            images = torch.from_numpy(normalize(raw, qlow, qhigh)).unsqueeze(0).to(device)
            with torch.autocast(device_type=device.type, enabled=amp_enabled): outputs = model(images)
            selected = [(t, outputs[0])] if t == 0 else []
            selected.append((t+1, outputs[1]))
            for frame, output in selected:
                pred = peaks(output, scale_zyx); gtf = gt.filter(pl.col("t")==frame).select(["z","y","x"]).to_numpy().astype(np.float32)
                truth = torch.from_numpy(gtf).to(device=device, dtype=torch.float32) / pred.new_tensor(cfg.downsample)
                matched_all.extend(match_sparse(pred, truth, scale_zyx))
                n_gt += len(truth); n_pred += len(pred)
        rows.append({"dataset": dataset_id, "frames": int(array.shape[0]), "sparse_gt_nodes": n_gt, "predicted_peaks": n_pred,
                     "mean_peaks_per_frame": n_pred / array.shape[0],
                     "matched_sparse_gt": len(matched_all), "sparse_recall": len(matched_all)/n_gt if n_gt else np.nan,
                     "mean_localisation_error_um": float(np.mean(matched_all)) if matched_all else np.nan,
                     "seconds": time.perf_counter() - dataset_started})
    frame = pd.DataFrame(rows); frame.to_csv(RUN / "detector_validation.csv", index=False)
    matched = frame.matched_sparse_gt.sum(); total_gt = frame.sparse_gt_nodes.sum()
    weighted_error = float((frame.mean_localisation_error_um.fillna(0) * frame.matched_sparse_gt).sum() / matched) if matched else np.nan
    summary = {"datasets": len(frame), "checkpoint": str(checkpoint), "sparse_gt_nodes": int(total_gt),
               "predicted_peaks": int(frame.predicted_peaks.sum()), "mean_peaks_per_frame": float(frame.predicted_peaks.sum() / frame.frames.sum()),
               "sparse_recall": float(matched/total_gt) if total_gt else np.nan,
               "mean_localisation_error_um": weighted_error, "validation_seconds": time.perf_counter() - validation_started,
               "peak_gpu_memory_gib": (torch.cuda.max_memory_allocated(device) / 2**30 if device.type == 'cuda' else 0.0),
               "note": "GT sparse: recall/error are diagnostic; unmatched peaks are not full detection false positives."}
    (RUN / "detector_validation_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return frame

if cfg.run_training:
    history = train_detector()
    display(history.tail())
elif cfg.run_validation:
    display(validate_detector())
else:
    print("Training и validation не запущены. Установите cfg.run_training=True; для независимой проверки также cfg.run_validation=True.")


## 8. Отчёт о запуске и кривые

Эта ячейка работает после training и validation, читает сохранённые артефакты и создаёт SVG-кривую без дополнительной зависимости matplotlib.


In [ ]:
from html import escape
from IPython.display import SVG, display

def line_path(values: np.ndarray, width: int, height: int, pad: int = 42) -> str:
    values = np.asarray(values, dtype=float); finite = np.isfinite(values)
    if finite.sum() < 2: return ""
    lo, hi = values[finite].min(), values[finite].max()
    if hi <= lo: hi = lo + 1.0
    xs = np.linspace(pad, width - pad, len(values)); ys = height - pad - (values - lo) / (hi - lo) * (height - 2 * pad)
    return " ".join(f"{x:.1f},{y:.1f}" for x, y in zip(xs[finite], ys[finite]))

def build_report_svg(step_history: pd.DataFrame, epoch_history: pd.DataFrame, validation_history: pd.DataFrame) -> str:
    width, height = 1000, 830; parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="white"/>', '<style>text{font:14px sans-serif;fill:#172033}.title{font-size:18px;font-weight:600}.axis{stroke:#9aa4b2}.grid{stroke:#e5e7eb}</style>']
    for origin_y, title, values, color in [(40, 'Train loss по шагам', step_history.loss.to_numpy() if len(step_history) else np.array([]), '#2563eb'), (310, 'Средний train loss по эпохам', epoch_history.mean_loss.to_numpy() if len(epoch_history) else np.array([]), '#dc2626'), (580, 'Validation sparse-GT recall по эпохам', validation_history.sparse_recall.to_numpy() if len(validation_history) else np.array([]), '#059669')]:
        chart_h = 210; pad = 42; parts += [f'<text class="title" x="42" y="{origin_y}">{escape(title)}</text>']
        for frac in (0, .5, 1): parts.append(f'<line class="grid" x1="42" y1="{origin_y+20+frac*(chart_h-50):.1f}" x2="958" y2="{origin_y+20+frac*(chart_h-50):.1f}"/>')
        path = line_path(values, 1000, chart_h, pad)
        if path: parts.append(f'<polyline fill="none" stroke="{color}" stroke-width="2" points="{path}" transform="translate(0,{origin_y+15})"/>')
        else: parts.append(f'<text x="42" y="{origin_y+110}">Нет данных: сначала включите cfg.run_training.</text>')
    return "".join(parts) + '</svg>'

step_history = pd.read_csv(RUN / 'train_history.csv') if (RUN / 'train_history.csv').exists() else pd.DataFrame()
epoch_history = pd.read_csv(RUN / 'epoch_history.csv') if (RUN / 'epoch_history.csv').exists() else pd.DataFrame()
validation = pd.read_csv(RUN / 'detector_validation.csv') if (RUN / 'detector_validation.csv').exists() else pd.DataFrame()
validation_history = pd.read_csv(RUN / 'validation_history.csv') if (RUN / 'validation_history.csv').exists() else pd.DataFrame()
summary_path = RUN / 'detector_validation_summary.json'
validation_summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {}
svg = build_report_svg(step_history, epoch_history, validation_history); (RUN / 'training_curves.svg').write_text(svg, encoding='utf-8'); display(SVG(svg))
display(epoch_history)
if len(validation_history): display(validation_history)
if len(validation): display(validation)
report = {'run_dir': str(RUN), 'config': asdict(cfg), 'train_steps': int(len(step_history)), 'epochs': epoch_history.to_dict(orient='records'), 'validation_by_epoch': validation_history.to_dict(orient='records'), 'validation_summary': validation_summary, 'artifacts': [p.name for p in sorted(RUN.iterdir())]}
(RUN / 'run_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print({'report': str(RUN / 'run_report.json'), 'curve': str(RUN / 'training_curves.svg'), 'validation': validation_summary or 'not run'})


## Дальше

После обучения detector: (1) validation на удержанном эмбрионе, (2) фиксируем detection threshold/NMS по этому fold, (3) запускаем второй этап — transformer/linker только по предсказанным детекциям, затем переходим к официальной graph-метрике.
